In [1]:
# A1. Cellule 1 : chargement
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
from shapely.geometry import LineString, MultiLineString
from osgeo import gdal
from scipy import ndimage as ndi
from skimage.draw import line as tracer
from skimage.transform import resize
from skimage.filters import threshold_otsu
from skimage.measure import find_contours
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

RACINE = r'C:\MonCarriere\tefeSAT\Hackaton_Sensat'
BASE = os.path.join(RACINE, 'CoastSat', 'data', 'SAINT_LOUIS_NORD')

cotes = gpd.read_file(os.path.join(RACINE, 'donnees', 'traits_de_cote.geojson')).to_crs(32628)
cotes['date'] = pd.to_datetime(cotes['date']).dt.strftime('%Y-%m-%d')
cotes['annee'] = cotes['date'].str[:4].astype(int)

def meilleure_date(annee):
    return cotes[cotes['annee'] == annee].sort_values('precision_m').iloc[0]['date']

DATE_ENTRAINEMENT = meilleure_date(2024)
DATES_TEST = [meilleure_date(a) for a in (2017, 2019, 2022, 2025)]
print('Date d\'entraînement :', DATE_ENTRAINEMENT)
print('Dates de test :', DATES_TEST)

Date d'entraînement : 2024-01-02
Dates de test : ['2017-02-07', '2019-01-08', '2022-01-02', '2025-01-06']


In [3]:
# A2. Cellule 2 : fonctions de lecture
fich_ref = glob.glob(os.path.join(BASE, '*reference_shoreline*.pkl'))[0]
ref = np.array(pickle.load(open(fich_ref, 'rb')))[:, :2]

def charger(date):
    f10 = sorted(glob.glob(os.path.join(BASE, 'S2', 'ms', date + '-*_ms.tif')))[0]
    f20 = os.path.join(BASE, 'S2', 'swir', os.path.basename(f10).replace('_ms.tif', '_swir.tif'))
    d = gdal.Open(f10)
    ms = np.stack([d.GetRasterBand(k + 1).ReadAsArray() for k in range(4)], axis=2).astype('float32') / 10000
    gt = d.GetGeoTransform()
    d20 = gdal.Open(f20)                      # garder la référence au dataset
    sw = d20.GetRasterBand(1).ReadAsArray().astype('float32') / 10000
    d20 = None                                # fermer proprement après lecture
    sw = resize(sw, ms.shape[:2], order=1, preserve_range=True).astype('float32')
    bleu, vert, rouge, pir = [ms[:, :, i] for i in range(4)]
    mndwi = (vert - sw) / (vert + sw + 1e-6)
    ndvi = (pir - rouge) / (pir + rouge + 1e-6)
    X = np.dstack([bleu, vert, rouge, pir, sw, mndwi, ndvi]).astype('float32')
    return X, gt, ms.sum(axis=2) > 0

def zone_proche(forme, gt, dist_max):
    ligne = np.zeros(forme, bool)
    cols = ((ref[:, 0] - gt[0]) / gt[1]).astype(int)
    rows = ((ref[:, 1] - gt[3]) / gt[5]).astype(int)
    for k in range(len(ref) - 1):
        rr, cc = tracer(rows[k], cols[k], rows[k + 1], cols[k + 1])
        ok = (rr >= 0) & (rr < forme[0]) & (cc >= 0) & (cc < forme[1])
        ligne[rr[ok], cc[ok]] = True
    return ndi.distance_transform_edt(~ligne) * abs(gt[1]) <= dist_max

X, gt, valide = charger(DATE_ENTRAINEMENT)
print('Image :', X.shape, '| pixels dans la zone de 100 m :', zone_proche(X.shape[:2], gt, 100).sum())

Image : (1008, 762, 7) | pixels dans la zone de 100 m : 30819


In [ ]:
# A3. Cellule 3 : tes clics d’apprentissage
%matplotlib qt
rgb = np.clip(X[:, :, [2, 1, 0]] * 3.5, 0, 1)
roi300 = zone_proche(X.shape[:2], gt, 300) & valide
lignes_roi = np.where(roi300.any(axis=1))[0]
bandes = np.array_split(np.arange(lignes_roi.min(), lignes_roi.max() + 1), 5)
clics = []
for k, b in enumerate(bandes):
    r0, r1 = b[0], b[-1] + 1
    cols = np.where(roi300[r0:r1].any(axis=0))[0]
    c0, c1 = max(cols.min() - 20, 0), min(cols.max() + 20, X.shape[1])
    fig, ax = plt.subplots(figsize=(7, 10))
    ax.imshow(rgb[r0:r1, c0:c1], extent=(c0, c1, r1, r0))
    for classe, texte, couleur in ((1, "6 clics EAU (dont 2 dans l'écume)", 'cyan'),
                                   (0, '6 clics SABLE (plage seulement)', 'yellow')):
        ax.set_title(f'Fenêtre {k + 1}/5 : {texte}', fontsize=12)
        fig.canvas.draw()
        pts = plt.ginput(n=6, timeout=0, show_clicks=True)
        for (x, y) in pts:
            clics.append((int(x), int(y), classe))
            ax.plot(x, y, 'o', color=couleur, ms=5)
    plt.close(fig)

pd.DataFrame(clics, columns=['col', 'row', 'classe']).assign(date=DATE_ENTRAINEMENT).to_csv(
    os.path.join(RACINE, 'donnees', 'points_apprentissage_rf.csv'), index=False)
print(len(clics), 'clics enregistrés')